# 🗂 Bases de datos vectoriales

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 14 · Bloque 4 — 45 minutos**

---

## Objetivo

Almacenar embeddings de texto en ChromaDB y realizar búsqueda semántica — el corazón de todo sistema RAG.

## Al terminar este bloque vas a poder:

1. Crear colecciones en ChromaDB y agregar documentos con metadata.
2. Buscar por similitud semántica y compararlo con búsqueda por palabras exactas.
3. Usar un modelo multilingüe para mejorar resultados en español argentino.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Base de datos vectorial** | Motor especializado en guardar y consultar embeddings con baja latencia. |
| **ChromaDB** | Base vectorial open-source embebida en Python — sin servidor externo. |
| **Similitud coseno** | Mide la cercanía entre dos vectores: 1 = idénticos, 0 = sin relación. |
| **Colección** | Grupo de documentos relacionados en ChromaDB, como una tabla en SQL. |
| **CRUD vectorial** | `add` / `get` / `update` / `delete` / `query` — las operaciones básicas. |

In [ ]:
!uv pip install -qq chromadb sentence-transformers

In [ ]:
# Verificamos que PyTorch está instalado (necesario para sentence-transformers)
import torch
print(f"PyTorch versión: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

## ¿Para qué sirve una base de datos vectorial?

### Analogía

Una base SQL busca por coincidencia exacta: si preguntás por 'asado', solo te devuelve registros que tengan la palabra 'asado'. Una base vectorial busca por significado: 'carne a las brasas', 'parrilla', 'bife' son conceptualmente cercanos a 'asado' aunque no compartan ninguna letra.

### Dónde vive esto en el mundo real

ChromaDB, Pinecone y Weaviate son la infraestructura invisible de los asistentes de documentos de Notion, Slack y GitHub Copilot. Cada vez que un chatbot empresarial responde con información de un manual específico, hay una base vectorial detrás recuperando los fragmentos más relevantes. En el próximo notebook vas a conectar ChromaDB con Gemini para construir exactamente eso.

### ✎ Para pensar

- ¿Por qué ChromaDB usa similitud coseno en lugar de distancia euclidiana para comparar embeddings?
- Si agregás un documento en inglés a una colección con textos en español, ¿el modelo base (`all-MiniLM`) lo va a comparar bien con consultas en español?

In [ ]:
import chromadb

# Creamos un cliente de ChromaDB en memoria
# Nota: Los datos se pierden al cerrar el notebook
# Para persistencia, usar: chromadb.PersistentClient(path="./chroma_db")
client = chromadb.Client()

print("ChromaDB inicializado correctamente")

## Demo: base de reviews de restaurantes porteños

Vamos a construir un recomendador semántico con reviews reales. El objetivo es que una consulta como 'algo barato para morfar' encuentre restaurantes relevantes aunque no usen esas palabras exactas.

In [ ]:
collection = client.get_or_create_collection(
    name="reviews_restaurantes"
)
print("Colección 'reviews_restaurantes' lista")

In [ ]:
# Reviews iniciales - variedad de restaurantes en CABA
reviews_iniciales = [
    "Fui a cenar pasta y la verdad que espectacular. Los ñoquis con salsa fileto increíbles. Precio razonable, como 8 lucas por persona. Ambiente tranquilo, perfecto para ir en pareja.",

    "La mejor pizza que comí en mi vida, no jodo. Masa finita, crocante, mucha muzzarella. Eso sí, siempre está lleno y hay que esperar. Vale la pena. Estilo porteño posta.",

    "Restaurante gourmet en Palermo, cocina de autor. Los platos son chicos pero re elaborados. Caro pero para una ocasión especial está bueno. Tiene buen vino también.",

    "Parrilla clásica de barrio. El bife de chorizo una locura, tierno y jugoso. Las papas fritas caseras. Atención familiar, te hacen sentir como en tu casa. Precios normales.",

    "Sushi delivery que pedimos seguido. Fresco, bien armado, llega rápido. No es el mejor que probé pero para el precio está más que bien. El combo para dos es suficiente."
]

# Metadatos: información adicional sobre cada review
metadatos_iniciales = [
    {"barrio": "San Telmo", "tipo": "italiana", "precio": "medio"},
    {"barrio": "Palermo", "tipo": "pizzeria", "precio": "medio"},
    {"barrio": "Palermo", "tipo": "gourmet", "precio": "alto"},
    {"barrio": "Villa Urquiza", "tipo": "parrilla", "precio": "medio"},
    {"barrio": "delivery", "tipo": "sushi", "precio": "medio"}
]

# IDs únicos para cada documento
ids_iniciales = ["review1", "review2", "review3", "review4", "review5"]

### Estructura de una colección

Cada `add()` necesita tres listas de igual longitud:

| Parámetro | Contenido |
|---|---|
| `documents` | Los textos en sí |
| `metadatas` | Diccionarios con datos extra (barrio, tipo, precio) |
| `ids` | Identificadores únicos para cada documento |

In [ ]:
# Agregamos las reviews a la colección
collection.add(
    documents=reviews_iniciales,
    metadatas=metadatos_iniciales,
    ids=ids_iniciales
)

print(f"Se agregaron {len(reviews_iniciales)} reviews a la base de datos")

In [ ]:
# Verificamos cuántos documentos tenemos
collection.count()

In [ ]:
# Usamos el método 'get' del objeto colección 'collection' para recuperar todos los documentos y metadatos indexados
todos = collection.get()

# Imprimimos un título informativo en la consola
print("Documentos en la colección:")

# Extraemos la lista de textos de los documentos de la colección desde el diccionario 'todos'
lista_documentos_todos = todos["documents"]

# Obtenemos la cantidad de documentos que se recuperaron usando la función 'len'
cantidad_documentos = len(lista_documentos_todos)

# Iteramos sobre el rango de la cantidad de documentos por su índice numérico para una lectura más didáctica
for idx in range(cantidad_documentos):
    # Obtenemos el texto de la review para el índice actual
    documento_actual = lista_documentos_todos[idx]
    
    # Calculamos la numeración visible sumando 1 al índice base 0 de Python
    numero_orden = idx + 1
    
    # Imprimimos la numeración y los primeros 80 caracteres de la review con puntos suspensivos
    print(f"\n{numero_orden}. {documento_actual[:80]}...")

## Búsqueda semántica vs búsqueda por palabras

Vas a comparar los dos tipos de búsqueda con los mismos datos para ver concretamente la diferencia.

In [ ]:
# Definimos una variable de tipo string con el texto de nuestra consulta semántica en español
consulta = "Busco un lugar para comer buena comida italiana, tipo ravioles o ñoquis"

# Usamos el método 'query' de la colección 'collection' para buscar los documentos más cercanos.
# El parámetro 'query_texts' recibe la lista con nuestra consulta y el parámetro 'n_results' define
# que queremos recuperar únicamente los 3 documentos con mayor similitud semántica en la base.
resultados = collection.query(
    query_texts=[consulta],
    n_results=3
)

# Imprimimos en pantalla la consulta realizada para referencia del usuario
print(f"CONSULTA: {consulta}")

# Imprimimos un encabezado estético para separar los resultados mostrados
print("\nREVIEWS MÁS SIMILARES:")
print("=" * 80)

# Extraemos la lista de textos de los documentos resultantes desde el diccionario 'resultados'.
# Dado que pasamos una sola consulta en la lista, los resultados están dentro del primer índice [0].
lista_documentos = resultados["documents"][0]

# Extraemos la lista de diccionarios de metadatos correspondientes desde el mismo diccionario 'resultados'.
# Al igual que con los documentos, accedemos al primer índice [0] de la lista devuelta.
lista_metadatos = resultados["metadatas"][0]

# Obtenemos la cantidad total de resultados devueltos usando la función incorporada 'len'
cantidad_resultados = len(lista_documentos)

# Recorremos cada uno de los resultados mediante un bucle indexado 'for' tradicional.
# Esto evita el uso de funciones complejas de una sola línea como 'zip' o 'enumerate' en la declaración del bucle.
for idx in range(cantidad_resultados):
    # Extraemos el texto del documento para el índice actual 'idx'
    documento_actual = lista_documentos[idx]
    
    # Extraemos el diccionario de metadatos para el índice actual 'idx'
    metadato_actual = lista_metadatos[idx]
    
    # Creamos el número de orden de visualización sumándole 1 al índice base 0 de Python
    numero_orden = idx + 1
    
    # Imprimimos el número de orden y el contenido de texto de la opinión en pantalla
    print(f"\n{numero_orden}. {documento_actual}")
    
    # Extraemos los campos específicos de metadatos (barrio, tipo y precio) y los imprimimos con sangría
    print(f"   Barrio: {metadato_actual['barrio']}, Tipo: {metadato_actual['tipo']}, Precio: {metadato_actual['precio']}")

In [ ]:
# Definimos una variable de tipo string con la segunda consulta semántica enfática en carne y precio
consulta2 = "Dónde puedo ir a comer buen asado sin gastar mucha plata?"

# Usamos el método 'query' del objeto colección 'collection' para buscar en base al significado del texto.
# Pasamos la consulta en una lista a 'query_texts' y solicitamos los 3 mejores resultados con 'n_results'.
resultados2 = collection.query(
    query_texts=[consulta2],
    n_results=3
)

# Mostramos la consulta en pantalla para referencia del estudiante
print(f"CONSULTA: {consulta2}")

# Imprimimos un separador de consola con fines estéticos y de ordenamiento de lectura
print("\nREVIEWS MÁS SIMILARES:")
print("=" * 80)

# Extraemos la lista de textos de los documentos devueltos por el query de ChromaDB para la primera consulta (índice 0)
lista_documentos2 = resultados2["documents"][0]

# Extraemos la lista de metadatos correspondientes a esos documentos en el mismo índice de consulta (índice 0)
lista_metadatos2 = resultados2["metadatas"][0]

# Medimos la cantidad de documentos devueltos usando el método estándar de Python 'len'
cantidad_resultados2 = len(lista_documentos2)

# Iteramos por índice a través de los resultados para mantener una sintaxis legible y explícita para quien aprende
for idx in range(cantidad_resultados2):
    # Obtenemos el fragmento de opinión correspondiente a la iteración actual
    documento_actual = lista_documentos2[idx]
    
    # Obtenemos los metadatos asociados a esa opinión en específico
    metadato_actual = lista_metadatos2[idx]
    
    # Calculamos el número correlativo sumando 1 al índice actual de la iteración
    numero_orden = idx + 1
    
    # Imprimimos en pantalla el número del resultado y el texto de la opinión gastronómica
    print(f"\n{numero_orden}. {documento_actual}")
    
    # Mostramos los campos individuales extraídos del diccionario de metadatos del restaurante
    print(f"   Barrio: {metadato_actual['barrio']}, Tipo: {metadato_actual['tipo']}, Precio: {metadato_actual['precio']}")

In [ ]:
# Búsqueda tradicional: buscar la palabra "asado" exacta en la base de datos usando el método 'get'
print("BÚSQUEDA TRADICIONAL (palabra exacta 'asado'):")

# Usamos el método 'get' de la colección 'collection' aplicando un filtro sobre el contenido del documento.
# El parámetro 'where_document' recibe un diccionario con el operador '$contains' para encontrar la palabra exacta.
busqueda_tradicional = collection.get(
    where_document={"$contains": "asado"}
)

# Extraemos la lista de documentos que coincidieron con el filtro de coincidencia exacta
lista_documentos_encontrados = busqueda_tradicional["documents"]

# Evaluamos si la lista contiene elementos midiendo su longitud con la función 'len'
if len(lista_documentos_encontrados) > 0:
    # Recorremos de forma simple y secuencial cada documento coincidente
    for doc in lista_documentos_encontrados:
        # Imprimimos el texto completo del documento
        print(f"- {doc}")
else:
    # Mostramos un mensaje explicativo en caso de que no haya coincidencia exacta
    print("No se encontraron resultados con la palabra 'asado'")

# Imprimimos separadores y textos conceptuales en pantalla para la comparación didáctica
print("\n" + "="*80)
print("\nBÚSQUEDA SEMÁNTICA (por significado):")
print("Encontró la parrilla aunque no use la palabra 'asado' exacta")

### ✎ Para pensar

- La búsqueda encontró 'ñoquis' ante la consulta 'comida italiana'. ¿Qué tiene que saber el modelo de embeddings para hacer esa conexión?
- Probá la consulta 'lugar barato para morfar' con el modelo base y el multilingüe. ¿Cuál da mejor resultado? ¿Por qué?

In [ ]:
# Nuevas reviews que llegan
nuevas_reviews = [
    "Bar de tragos en Palermo re copado. Buenos cócteles, música en vivo los fines de semana. Se llena bastante después de las 11. Tiene terraza.",

    "Cafetería de especialidad, café re rico, tienen opciones veganas. Ambiente tranquilo para trabajar con la compu. WiFi gratis y enchufes.",

    "Bodegón español tradicional. Las croquetas y la tortilla de papa son de otro planeta. Vino de la casa riquísimo. Porteño viejo, 100 años de historia.",

    "Hamburguesería gourmet. Las papas con cheddar y bacon son adictivas. Burgers de 200gr, jugosas. Podes armar la tuya. Delivery hasta las 2am.",

    "Cantina familiar muy casera. La comida es simple pero rica, como la que hace tu nona. Milanesas gigantes. Super económico, 6 lucas comes re bien."
]

nuevos_metadatos = [
    {"barrio": "Palermo", "tipo": "bar", "precio": "medio-alto"},
    {"barrio": "Colegiales", "tipo": "cafeteria", "precio": "medio"},
    {"barrio": "Montserrat", "tipo": "española", "precio": "medio"},
    {"barrio": "Belgrano", "tipo": "hamburguesas", "precio": "medio"},
    {"barrio": "Boedo", "tipo": "casera", "precio": "bajo"}
]

In [ ]:
# Función helper para agregar reviews de forma organizada
def agregar_reviews(collection, reviews, metadatos, prefijo_id="review"):
    """
    Agrega nuevas reviews a una colección existente.

    Parámetros:
    -----------
    collection : chromadb.Collection
        La colección donde agregar los documentos
    reviews : list
        Lista de textos de reviews
    metadatos : list
        Lista de diccionarios con metadatos
    prefijo_id : str
        Prefijo para generar IDs únicos
    """
    # Obtenemos la cantidad de documentos que ya existen en la colección
    # usando el método 'count' del objeto de colección.
    count_actual = collection.count()
    
    # Creamos una lista vacía para ir guardando los IDs generados secuencialmente
    nuevos_ids = []
    
    # Obtenemos la cantidad de nuevas reviews a procesar usando 'len'
    cantidad_nuevas = len(reviews)
    
    # Usamos un bucle para generar un ID único para cada nuevo documento
    for idx_id in range(cantidad_nuevas):
        # Calculamos el número de orden sumando el recuento actual, el índice de iteración y 1
        numero_id = count_actual + idx_id + 1
        
        # Formateamos el ID uniendo el prefijo de texto y el número calculado
        id_unico = f"{prefijo_id}{numero_id}"
        
        # Agregamos el ID generado a nuestra lista con el método 'append'
        nuevos_ids.append(id_unico)

    # Llamamos al método 'add' de la colección para indexar los nuevos textos,
    # metadatos y la lista de IDs que generamos secuencialmente.
    collection.add(
        documents=reviews,
        metadatas=metadatos,
        ids=nuevos_ids
    )

    # Imprimimos mensajes pedagógicos de confirmación para el estudiante
    print(f"Se agregaron {cantidad_nuevas} nuevas reviews")
    print(f"Total de documentos en la colección: {collection.count()}")
    
    # Devolvemos la lista de IDs generados
    return nuevos_ids

In [ ]:
# Agregamos las nuevas reviews
ids_nuevos = agregar_reviews(collection, nuevas_reviews, nuevos_metadatos)

## Mejorando con embeddings multilingüe

El modelo por defecto (`all-MiniLM-L6-v2`) fue entrenado mayormente en inglés. Para modismos como 're copado', 'posta', 'morfar', necesitamos `multilingual-e5-large`.

**Trade-off:** más preciso para español, pero más lento y ocupa más memoria.

In [ ]:
# Importamos el módulo 'embedding_functions' de la librería 'chromadb.utils'.
# Este módulo provee funciones utilitarias preparadas para integrar distintos modelos de embeddings (como OpenAI, Cohere, o locales como SentenceTransformers) en ChromaDB.
from chromadb.utils import embedding_functions

# Instanciamos la clase 'SentenceTransformerEmbeddingFunction' del módulo de ChromaDB.
# Esta clase permite calcular embeddings de manera local usando modelos de la biblioteca 'sentence-transformers'.
# Pasamos el argumento 'model_name' configurado con 'intfloat/multilingual-e5-large', que es un modelo multilingüe
# de última generación en Hugging Face, ideal para entender el español rioplatense y sus modismos.
# Nota: La primera vez que corra esta celda se va a descargar el archivo del modelo (~1.1 GB).
modelo_multilenguaje = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="intfloat/multilingual-e5-large"
)

# Mostramos un mensaje de confirmación pedagógico indicando que el modelo se cargó exitosamente en memoria
print("✓ Modelo multilingüe 'multilingual-e5-large' cargado y listo en memoria")

In [ ]:
# Usamos el método 'get_or_create_collection' del objeto cliente 'client' de ChromaDB.
# Este método crea una colección nueva con el nombre especificado, o la recupera si ya existe.
# Le pasamos el parámetro 'name' para nombrar la colección como 'reviews_multilenguaje'
# y el parámetro 'embedding_function' configurado con 'modelo_multilenguaje' para indicarle a ChromaDB
# que debe usar nuestro modelo en lugar del predeterminado en inglés ('all-MiniLM-L6-v2') al procesar los textos.
collection_mejorada = client.get_or_create_collection(
    name="reviews_multilenguaje",
    embedding_function=modelo_multilenguaje
)

# Imprimimos mensajes informativos para verificar el estado de la colección y su ventaja didáctica
print("✓ Colección 'reviews_multilenguaje' lista con modelo multilenguaje")
print("  Esta colección entiende modismos rioplatenses mucho mejor que la base")

In [ ]:
# Usamos el método 'add' del objeto 'collection_mejorada' para indexar nuestro lote inicial de documentos.
# Pasamos el parámetro 'documents' con la lista 'reviews_iniciales' que contiene los textos de las opiniones,
# el parámetro 'metadatas' con la lista de diccionarios 'metadatos_iniciales' (barrio, tipo de cocina y nivel de precio),
# y el parámetro 'ids' con los identificadores únicos de cada documento en 'ids_iniciales'.
collection_mejorada.add(
    documents=reviews_iniciales,
    metadatas=metadatos_iniciales,
    ids=ids_iniciales
)

# Agregamos el segundo lote de reviews (las nuevas opiniones) usando la función helper 'agregar_reviews'.
# Pasamos 'collection_mejorada' como el destino, la lista 'nuevas_reviews' con los textos,
# y la lista 'nuevos_metadatos' con sus respectivos diccionarios de metadatos.
# Esta función calcula automáticamente los IDs incrementales correlativos correspondientes.
ids_nuevos_mejorada = agregar_reviews(
    collection=collection_mejorada,
    reviews=nuevas_reviews,
    metadatos=nuevos_metadatos
)

# Usamos el método 'count' de 'collection_mejorada' para verificar y mostrar cuántos documentos totales tiene cargados.
total_documentos = collection_mejorada.count()
print(f"✓ Colección mejorada poblada. Total de documentos indexados: {total_documentos}")

In [ ]:
# Definimos una variable de tipo string con una consulta cargada de lunfardo argentino
consulta_argentina = "Busco un lugar re copado para morfar algo barato y llenadero"

# Imprimimos líneas decorativas y el texto de la consulta en la consola para una lectura prolija
print("="*80)
print(f"CONSULTA: {consulta_argentina}")
print("="*80)

# Imprimimos el encabezado para el bloque del modelo base en inglés
print("\n1. MODELO BASE (all-MiniLM-L6-v2):")
print("-"*80)

# Usamos el método 'query' en la colección inicial 'collection' (que usa el modelo base por defecto)
# pasándole el texto de consulta y limitando la recuperación a los 3 mejores resultados semánticos.
resultado_base = collection.query(
    query_texts=[consulta_argentina],
    n_results=3
)

# Extraemos la lista de textos de los documentos devueltos en la primera posición de resultados (índice 0)
lista_documentos_base = resultado_base["documents"][0]

# Calculamos la cantidad de documentos devueltos por la consulta semántica base
cantidad_base = len(lista_documentos_base)

# Recorremos de manera explícita y didáctica cada documento recuperado por el modelo base
for idx in range(cantidad_base):
    # Extraemos el texto de la review gastronómica para el índice actual
    documento_actual = lista_documentos_base[idx]
    
    # Creamos la numeración sumando 1 al índice numérico de Python
    numero_orden = idx + 1
    
    # Mostramos los primeros 100 caracteres de la opinión agregando puntos suspensivos para abreviarla en pantalla
    print(f"\n{numero_orden}. {documento_actual[:100]}...")

# Imprimimos un separador visual antes de mostrar los resultados del segundo modelo
print("\n" + "="*80)

# Imprimimos el encabezado explicativo para el bloque del modelo multilingüe optimizado
print("\n2. MODELO MULTILENGUAJE (multilingual-e5-large):")
print("-"*80)

# Usamos el método 'query' sobre la colección 'collection_mejorada' que fue inicializada
# pasándole la función de embedding multilingüe para procesar adecuadamente los modismos locales.
resultado_mejorado = collection_mejorada.query(
    query_texts=[consulta_argentina],
    n_results=3
)

# Extraemos la lista de documentos de la consulta para el modelo multilingüe (índice 0)
lista_documentos_mejorado = resultado_mejorado["documents"][0]

# Extraemos los diccionarios de metadatos correspondientes a esos documentos (índice 0)
lista_metadatos_mejorado = resultado_mejorado["metadatas"][0]

# Medimos cuántos resultados arrojó la consulta en la colección mejorada
cantidad_mejorada = len(lista_documentos_mejorado)

# Recorremos ordenadamente por índice el resultado multilingüe para simplificar la lectura cognitiva
for idx in range(cantidad_mejorada):
    # Obtenemos el texto del documento para el índice actual
    documento_actual = lista_documentos_mejorado[idx]
    
    # Obtenemos la información de metadatos del restaurante en el índice actual
    metadato_actual = lista_metadatos_mejorado[idx]
    
    # Calculamos la numeración visible sumando 1
    numero_orden = idx + 1
    
    # Mostramos los primeros 100 caracteres del texto recuperado y sus metadatos clasificados en consola
    print(f"\n{numero_orden}. {documento_actual[:100]}...")
    print(f"   Tipo: {metadato_actual['tipo']}, Precio: {metadato_actual['precio']}")

### 🐛 Laboratorio de Rotura

El código de abajo *parece* correcto. Antes de ejecutarlo, predecí: ¿qué va a pasar si intentás crear una colección con un nombre que ya existe?

```python
colision = client.create_collection(name="reviews_restaurantes")
```

Ejecutá la celda siguiente.

- ¿Qué esperabas?
- ¿Qué pasó en realidad?
- ¿Cómo deberías manejar esto en un sistema real que se reinicia?

No mires la explicación todavía.

In [ ]:
# ── Momento 1 — La colisión de nombres ──
# Usamos un bloque 'try-except' de Python para capturar de forma segura la excepción que esperamos que ocurra
try:
    # Intentamos forzar la creación de una colección con un nombre que ya existe usando el método 'create_collection'
    # del objeto cliente 'client'. Esto debería fallar porque los nombres de colecciones deben ser únicos.
    colision = client.create_collection(name="reviews_restaurantes")
    
    # Si no ocurre un error (lo cual no debería pasar), imprimimos este mensaje explicativo
    print("Se creó... pero ¿cómo puede haber dos colecciones con el mismo nombre?")
    
except Exception as e:
    # Capturamos cualquier excepción (de tipo Exception) que lance ChromaDB ante la colisión de nombres
    # Imprimimos la descripción del error capturado usando el objeto excepción 'e'
    print(f"✗ Error: {e}")
    print()
    
    # Imprimimos un mensaje aclaratorio pedagógico sobre el comportamiento esperado de ChromaDB
    print("ChromaDB no permite dos colecciones con el mismo nombre en el mismo cliente.")

# ── Momento 2 — La solución idiomática ──
# Usamos el método 'get_or_create_collection' del cliente 'client' de ChromaDB para resolver la colisión.
# Este método verifica si la colección 'reviews_restaurantes' ya existe: si es así, la devuelve;
# si no existe en el cliente, la crea de forma segura sin lanzar errores de ejecución.
collection_segura = client.get_or_create_collection(name="reviews_restaurantes")

# Usamos el método 'count' de la colección 'collection_segura' para obtener la cantidad de elementos en ella
cantidad_docs = collection_segura.count()
print(f"\n✓ get_or_create_collection: {cantidad_docs} documentos ya cargados")

# Imprimimos un consejo práctico de desarrollo de software para el uso de esta función en entornos reales
print("   Usá get_or_create_collection en producción para hacer el sistema resistente a reinicios.")

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# Probá las siguientes búsquedas y compará los resultados entre el modelo base
# y el modelo multilingüe:
#   1. 'Un lugar romántico para ir con mi pareja'
#   2. 'Algo para comer rápido al mediodía'
#   3. 'Comida como la que hace la abuela'
#
# Para cada una, ejecutá la consulta en ambas colecciones y verificá cuál de
# los modelos devuelve resultados con mejor sentido semántico y contexto local.
#
# ─────────────────────────────────────────────────────────────────────────────

# Definimos una variable de tipo string con la consulta que elegimos probar
MI_CONSULTA = "Un lugar romántico para ir con mi pareja"
print(f"Consulta: {MI_CONSULTA}\n")

# -----------------------------------------------------------------------------
# PARTE 1: Consulta usando el Modelo Base (all-MiniLM-L6-v2)
# -----------------------------------------------------------------------------

print("=== BÚSQUEDA CON MODELO BASE ===")

# Usamos el método 'query' en la colección 'collection' (que tiene el modelo base)
# para buscar los 3 mejores resultados semánticos para nuestra consulta de prueba.
resultados_base = collection.query(
    query_texts=[MI_CONSULTA],
    n_results=3
)

# Extraemos la lista de textos de los documentos devueltos (primer índice [0] del query)
documentos_base = resultados_base["documents"][0]

# Extraemos los diccionarios de metadatos correspondientes (primer índice [0] del query)
metadatos_base = resultados_base["metadatas"][0]

# Obtenemos la cantidad de resultados para iterar por índice
cantidad_base = len(documentos_base)

# Iteramos por índice a través de los resultados para mantener un flujo de lectura sencillo y explícito
for idx in range(cantidad_base):
    # Extraemos el documento de la iteración actual
    doc_actual = documentos_base[idx]
    
    # Extraemos el metadato de la iteración actual
    meta_actual = metadatos_base[idx]
    
    # Mostramos los metadatos relevantes (tipo de comida y barrio) junto al texto resumido en pantalla
    print(f"  {meta_actual['tipo']} ({meta_actual['barrio']}): {doc_actual[:80]}...")

print("\n")

# -----------------------------------------------------------------------------
# PARTE 2: Consulta usando el Modelo Multilingüe (multilingual-e5-large)
# -----------------------------------------------------------------------------

print("=== BÚSQUEDA CON MODELO MULTILENGUAJE ===")

# Usamos el método 'query' en 'collection_mejorada' (que tiene el modelo multilingüe)
# para buscar los 3 mejores resultados semánticos para nuestra consulta de prueba.
resultados_multi = collection_mejorada.query(
    query_texts=[MI_CONSULTA],
    n_results=3
)

# Extraemos la lista de textos de los documentos devueltos (primer índice [0] del query)
documentos_multi = resultados_multi["documents"][0]

# Extraemos los diccionarios de metadatos correspondientes (primer índice [0] del query)
metadatos_multi = resultados_multi["metadatas"][0]

# Obtenemos la cantidad de resultados para iterar por índice
cantidad_multi = len(documentos_multi)

# Iteramos por índice a través de los resultados en la colección mejorada
for idx in range(cantidad_multi):
    # Extraemos el documento de la iteración actual
    doc_actual = documentos_multi[idx]
    
    # Extraemos el metadato de la iteración actual
    meta_actual = metadatos_multi[idx]
    
    # Mostramos los metadatos relevantes y el fragmento del documento multilingüe
    print(f"  {meta_actual['tipo']} ({meta_actual['barrio']}): {doc_actual[:80]}...")

> **✎ Mentor reversible (bonus).** Hasta acá fuiste quien aprende. Ahora dalo vuelta: explicale en treinta segundos, en voz alta o por escrito, la diferencia entre búsqueda semántica y búsqueda por palabras exactas a alguien que nunca escuchó hablar de vectores. Usá el ejemplo del asado si te ayuda. Si esa persona imaginaria lo entiende, es porque vos ya lo entendiste de verdad.

## Vista previa: de la búsqueda a RAG

Lo que hiciste hasta acá es la mitad de RAG: la parte de **Retrieval** (recuperación).

| Paso | Qué hace | Estado |
|---|---|---|
| Guardar documentos con embeddings | ChromaDB.add() | ✅ hecho |
| Buscar los más relevantes | ChromaDB.query() | ✅ hecho |
| Pasarlos como contexto al LLM | prompt con contexto | próximo notebook |
| Generar respuesta fundamentada | Gemini | próximo notebook |

## Cierre del bloque

| Concepto | Qué aprendiste |
|---|---|
| **ChromaDB** | Base vectorial embebida, sin servidor externo |
| **Colección** | Grupo de documentos con embeddings, metadatos e IDs |
| **Búsqueda semántica** | Encuentra por significado, no por palabras exactas |
| **Multilingual-e5** | Modelo de embeddings optimizado para español regional |
| **CRUD vectorial** | add / get / update / delete / query |

**Próximo bloque →** RAG completo: vas a conectar ChromaDB con Gemini y construir un sistema que responde preguntas sobre tus propios documentos.